# VGGish + HuBERT + Whisper-medium - SVM

**Идея:** объединить эмбеддинги трёх разнородных моделей в один вектор и обучить SVM.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import make_scorer, f1_score
from transformers import HubertModel, WhisperModel, WhisperProcessor

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate
from shared.data_utils import build_feature_matrix, load_audio
from shared.results_utils import save_result_csv

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


In [2]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

print(f"Train+Val: {len(paths_trainval)}  good: {(labels_trainval==0).sum()}, bad: {(labels_trainval==1).sum()}")
print(f"Test:      {len(paths_test)}  good: {(labels_test==0).sum()},  bad: {(labels_test==1).sum()}")

Train+Val: 2356  good: 1594, bad: 762
Test:      416  good: 281,  bad: 135


## VGGish embeddings (mean + std + max - 384 dim)

In [5]:
vggish_model = torch.hub.load("harritaylor/torchvggish", "vggish")
vggish_model.eval().to(DEVICE)

def extract_vggish(path: str) -> np.ndarray:
    with torch.no_grad():
        emb = vggish_model.forward(path).cpu().numpy()
    if emb.ndim == 1:
        emb = emb[np.newaxis, :]
    return np.concatenate([emb.mean(0), emb.std(0), emb.max(0)]).astype(np.float32)

print("VGGish train+val...")
X_vggish_trainval = build_feature_matrix(paths_trainval, extract_vggish, n_jobs=1)
print("VGGish test...")
X_vggish_test     = build_feature_matrix(paths_test,     extract_vggish, n_jobs=1)

del vggish_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"VGGish shape: {X_vggish_trainval.shape}")

Using cache found in /Users/dk/.cache/torch/hub/harritaylor_torchvggish_master


VGGish train+val...
VGGish test...
VGGish shape: (2356, 384)


## HuBERT embeddings (mean + std + max - 2304 dim)

In [6]:
SR      = config.TARGET_SR
MAX_LEN = int(SR * config.MAX_DURATION_SEC)

hubert_model = HubertModel.from_pretrained("facebook/hubert-base-ls960").to(DEVICE)
hubert_model.eval()

def extract_hubert(path: str) -> np.ndarray:
    y, _ = load_audio(path, sr=SR)
    wav = torch.from_numpy(
        np.pad(y, (0, max(0, MAX_LEN - len(y))))[:MAX_LEN]
    ).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        hs = hubert_model(wav).last_hidden_state[0].cpu().numpy()  # (T, 768)
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

print("HuBERT train+val...")
X_hubert_trainval = build_feature_matrix(paths_trainval, extract_hubert, n_jobs=1)
print("HuBERT test...")
X_hubert_test     = build_feature_matrix(paths_test,     extract_hubert, n_jobs=1)

del hubert_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"HuBERT shape: {X_hubert_trainval.shape}")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

HuBERT train+val...
HuBERT test...
HuBERT shape: (2356, 2304)


## Whisper-medium embeddings (mean + std + max - 3072 dim)

In [7]:
WHISPER_ID = "openai/whisper-medium"
processor  = WhisperProcessor.from_pretrained(WHISPER_ID)
whisper    = WhisperModel.from_pretrained(WHISPER_ID).to(DEVICE)
whisper.eval()
WHISPER_DIM = whisper.config.d_model  # 1024

def extract_whisper(path: str) -> np.ndarray:
    y, _ = load_audio(path, sr=16000)
    feats = processor.feature_extractor(y, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        hs = whisper.encoder(
            input_features=feats.input_features.to(DEVICE)
        ).last_hidden_state[0].cpu().numpy()  # (T, 1024)
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

print("Whisper-medium train+val...")
X_whisper_trainval = build_feature_matrix(paths_trainval, extract_whisper, n_jobs=1)
print("Whisper-medium test...")
X_whisper_test     = build_feature_matrix(paths_test,     extract_whisper, n_jobs=1)

del whisper
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"Whisper shape: {X_whisper_trainval.shape}")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Whisper-medium train+val...
Whisper-medium test...
Whisper shape: (2356, 3072)


## Конкатенация и SVM

In [8]:
X_trainval = np.hstack([X_vggish_trainval, X_hubert_trainval, X_whisper_trainval, letters_trainval])
X_test     = np.hstack([X_vggish_test,     X_hubert_test,     X_whisper_test,     letters_test])

EMBED_DIM = X_vggish_trainval.shape[1] + X_hubert_trainval.shape[1] + X_whisper_trainval.shape[1]
print(f"Итоговый вектор (без letters): {EMBED_DIM}-dim")
print(f"Итоговый вектор (с letters):   {X_trainval.shape[1]}-dim")

Итоговый вектор (без letters): 5760-dim
Итоговый вектор (с letters):   5770-dim


In [9]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
f1_bad_scorer = make_scorer(f1_score, pos_label=config.CLASS_BAD, average="binary")
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}

t0 = time.perf_counter()
grid = GridSearchCV(pipeline, param_grid, cv=5,
                    scoring=f1_bad_scorer, refit=True, n_jobs=-1)
grid.fit(X_trainval, labels_trainval)
train_time_sec = time.perf_counter() - t0

best_params = grid.best_params_
best_clf    = grid.best_estimator_
print(f"Лучшие параметры: {best_params}")
print(f"Время GridSearch: {train_time_sec:.1f} с")

Лучшие параметры: {'clf__C': 1.0, 'clf__gamma': 'scale'}
Время GridSearch: 2271.2 с


## Оценка на test

In [12]:
y_proba_oof = cross_val_predict(
    best_clf, X_trainval, labels_trainval, cv=5, method="predict_proba"
)[:, config.CLASS_BAD]
threshold = find_optimal_threshold(labels_trainval, y_proba_oof)

y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=threshold, verbose=True)

pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_embed_fusion_svm",
    experiment_name="Fusion embeddings: VGGish + HuBERT + Whisper-medium → SVM",
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=EMBED_DIM,
    embed_dim_note="VGGish(384 mean+std+max) + HuBERT(2304 mean+std+max) + Whisper-medium(3072 mean+std+max)",
    notes=(
        f"GridSearchCV cv=5 + threshold | "
        f"best_params={best_params} | thr={threshold:.2f}"
    ),
    train_time_sec=train_time_sec,
)

              precision    recall  f1-score   support

        good       0.87      0.90      0.88       281
         bad       0.77      0.72      0.74       135

    accuracy                           0.84       416
   macro avg       0.82      0.81      0.81       416
weighted avg       0.84      0.84      0.84       416

Threshold : 0.36
Accuracy  : 0.8389
F1-macro  : 0.8130
F1-bad    : 0.7433
ROC-AUC   : 0.9079


PosixPath('/Users/dk/Desktop/ВШЭ/ВКР/HSE_VKR_DetectingSpeechDefects/experiments/05_ensembles/exp_embed_fusion_svm/result.csv')